# Model Experiments on Balanced Normalized Data

This notebook runs all classification models on the GAN-augmented balanced normalized dataset.

## Models:
1. Random Forest Classifier
2. Gradient Boosting Classifier
3. XGBoost Classifier
4. CatBoost Classifier
5. Logistic Regression
6. Artificial Neural Network (PyTorch)

## Target: CGPA3_Class (3 classes)
- Class 0 (normalized: 0.0) = CGPA 3.5-4.0
- Class 1 (normalized: 0.5) = CGPA 3.0-3.49
- Class 2 (normalized: 1.0) = CGPA <3.0

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

# Deep Learning - PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Train-Test Split and Metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, balanced_accuracy_score, roc_auc_score
)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("All libraries imported successfully!")

## 2. Load Balanced Normalized Data

In [ ]:
# Load the GAN-augmented balanced normalized data
df = pd.read_csv(r'../data/Final_Combined_Normalized.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
# Check class distribution
TARGET_COLUMN = 'CGPA3_Class'

print("Class Distribution (Normalized Values):")
print(df[TARGET_COLUMN].value_counts().sort_index())

# Convert normalized target back to integer classes for modeling
# 0.0 -> 0, 0.5 -> 1, 1.0 -> 2
df['Target'] = (df[TARGET_COLUMN] * 2).round().astype(int)

print("\nConverted Target Classes:")
print(df['Target'].value_counts().sort_index())
print("\n0 = CGPA 3.5-4.0 (High)")
print("1 = CGPA 3.0-3.49 (Average)")
print("2 = CGPA <3.0 (Low)")

## 3. Prepare Features and Target

In [ ]:
# Define features (exclude target columns)
exclude_cols = ['CGPA3_Class', 'Target', 'Current_CGPA5']  # Exclude target and related columns
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"Number of features: {len(feature_cols)}")
print(f"Features: {feature_cols}")

In [ ]:
# Prepare X and y
X = df[feature_cols].values
y = df['Target'].values

# Train-Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u}: {c} ({c/len(y_train)*100:.1f}%)")

print(f"\nTest class distribution:")
unique, counts = np.unique(y_test, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u}: {c} ({c/len(y_test)*100:.1f}%)")

## 4. Helper Functions

In [ ]:
# Store all results
all_results = []

def evaluate_model(model, X_test, y_test, model_name):
    """
    Evaluate model and return metrics
    """
    y_pred = model.predict(X_test)
    
    # For models that support predict_proba
    try:
        y_prob = model.predict_proba(X_test)
        roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
    except:
        roc_auc = None
    
    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    
    result = {
        'Model': model_name,
        'Accuracy': round(accuracy, 4),
        'Macro F1-Score': round(macro_f1, 4),
        'Balanced Accuracy': round(balanced_acc, 4),
        'ROC-AUC': round(roc_auc, 4) if roc_auc else 'N/A'
    }
    
    all_results.append(result)
    return result, y_pred

def plot_confusion_matrix(y_test, y_pred, model_name):
    """
    Plot confusion matrix
    """
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['High (3.5-4.0)', 'Avg (3.0-3.49)', 'Low (<3.0)'],
                yticklabels=['High (3.5-4.0)', 'Avg (3.0-3.49)', 'Low (<3.0)'])
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()

def print_classification_report(y_test, y_pred, model_name):
    """
    Print classification report
    """
    print(f"\n{'='*60}")
    print(f"Classification Report - {model_name}")
    print('='*60)
    print(classification_report(y_test, y_pred, 
          target_names=['High (3.5-4.0)', 'Avg (3.0-3.49)', 'Low (<3.0)']))

---
## 5. Model 1: Random Forest Classifier

In [ ]:
print("Training Random Forest Classifier...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
result, y_pred_rf = evaluate_model(rf_model, X_test, y_test, 'Random Forest')

print(f"\nResults:")
for key, value in result.items():
    print(f"  {key}: {value}")

In [ ]:
plot_confusion_matrix(y_test, y_pred_rf, 'Random Forest')
print_classification_report(y_test, y_pred_rf, 'Random Forest')

---
## 6. Model 2: Gradient Boosting Classifier

In [ ]:
print("Training Gradient Boosting Classifier...")

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

gb_model.fit(X_train, y_train)
result, y_pred_gb = evaluate_model(gb_model, X_test, y_test, 'Gradient Boosting')

print(f"\nResults:")
for key, value in result.items():
    print(f"  {key}: {value}")

In [ ]:
plot_confusion_matrix(y_test, y_pred_gb, 'Gradient Boosting')
print_classification_report(y_test, y_pred_gb, 'Gradient Boosting')

---
## 7. Model 3: XGBoost Classifier

In [ ]:
print("Training XGBoost Classifier...")

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

xgb_model.fit(X_train, y_train)
result, y_pred_xgb = evaluate_model(xgb_model, X_test, y_test, 'XGBoost')

print(f"\nResults:")
for key, value in result.items():
    print(f"  {key}: {value}")

In [ ]:
plot_confusion_matrix(y_test, y_pred_xgb, 'XGBoost')
print_classification_report(y_test, y_pred_xgb, 'XGBoost')

---
## 8. Model 4: CatBoost Classifier

In [ ]:
print("Training CatBoost Classifier...")

catboost_model = CatBoostClassifier(
    iterations=100,
    depth=5,
    learning_rate=0.1,
    random_state=42,
    verbose=False
)

catboost_model.fit(X_train, y_train)
result, y_pred_cat = evaluate_model(catboost_model, X_test, y_test, 'CatBoost')

print(f"\nResults:")
for key, value in result.items():
    print(f"  {key}: {value}")

In [ ]:
plot_confusion_matrix(y_test, y_pred_cat, 'CatBoost')
print_classification_report(y_test, y_pred_cat, 'CatBoost')

---
## 9. Model 5: Logistic Regression

In [ ]:
print("Training Logistic Regression...")

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    multi_class='multinomial',
    solver='lbfgs'
)

lr_model.fit(X_train, y_train)
result, y_pred_lr = evaluate_model(lr_model, X_test, y_test, 'Logistic Regression')

print(f"\nResults:")
for key, value in result.items():
    print(f"  {key}: {value}")

In [ ]:
plot_confusion_matrix(y_test, y_pred_lr, 'Logistic Regression')
print_classification_report(y_test, y_pred_lr, 'Logistic Regression')

---
## 10. Model 6: Artificial Neural Network (PyTorch)

In [ ]:
# Define Neural Network Architecture
class NeuralNetwork(nn.Module):
    def __init__(self, input_size, hidden_sizes, num_classes):
        super(NeuralNetwork, self).__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

print("Neural Network class defined.")

In [ ]:
# Prepare data for PyTorch
X_train_tensor = torch.FloatTensor(X_train).to(device)
y_train_tensor = torch.LongTensor(y_train).to(device)
X_test_tensor = torch.FloatTensor(X_test).to(device)
y_test_tensor = torch.LongTensor(y_test).to(device)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Initialize model
input_size = X_train.shape[1]
hidden_sizes = [128, 64, 32]
num_classes = 3

ann_model = NeuralNetwork(input_size, hidden_sizes, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(ann_model.parameters(), lr=0.001)

print(f"Model architecture:")
print(ann_model)

In [ ]:
# Training loop
print("Training Neural Network...")
epochs = 100
train_losses = []

for epoch in range(epochs):
    ann_model.train()
    epoch_loss = 0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = ann_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / len(train_loader))
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_losses[-1]:.4f}")

print("\nTraining completed!")

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses)
plt.title('Neural Network Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

In [ ]:
# Evaluate Neural Network
ann_model.eval()
with torch.no_grad():
    outputs = ann_model(X_test_tensor)
    _, y_pred_ann = torch.max(outputs, 1)
    y_pred_ann = y_pred_ann.cpu().numpy()
    
    # Get probabilities for ROC-AUC
    y_prob_ann = torch.softmax(outputs, dim=1).cpu().numpy()

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred_ann)
macro_f1 = f1_score(y_test, y_pred_ann, average='macro')
balanced_acc = balanced_accuracy_score(y_test, y_pred_ann)
roc_auc = roc_auc_score(y_test, y_prob_ann, multi_class='ovr', average='macro')

result = {
    'Model': 'Neural Network',
    'Accuracy': round(accuracy, 4),
    'Macro F1-Score': round(macro_f1, 4),
    'Balanced Accuracy': round(balanced_acc, 4),
    'ROC-AUC': round(roc_auc, 4)
}
all_results.append(result)

print(f"\nNeural Network Results:")
for key, value in result.items():
    print(f"  {key}: {value}")

In [ ]:
plot_confusion_matrix(y_test, y_pred_ann, 'Neural Network')
print_classification_report(y_test, y_pred_ann, 'Neural Network')

---
## 11. Model Comparison

In [ ]:
# Create comparison dataframe
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)

print("="*80)
print("MODEL COMPARISON - Balanced Normalized Data (GAN Augmented)")
print("="*80)
print(results_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['Accuracy', 'Macro F1-Score', 'Balanced Accuracy', 'ROC-AUC']
colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c']

for idx, (ax, metric, color) in enumerate(zip(axes.flat, metrics, colors)):
    if metric == 'ROC-AUC':
        # Handle 'N/A' values
        plot_df = results_df[results_df['ROC-AUC'] != 'N/A'].copy()
        plot_df['ROC-AUC'] = plot_df['ROC-AUC'].astype(float)
        values = plot_df[metric].values
        models = plot_df['Model'].values
    else:
        values = results_df[metric].values
        models = results_df['Model'].values
    
    bars = ax.barh(models, values, color=color, alpha=0.7)
    ax.set_xlabel(metric)
    ax.set_title(f'{metric} by Model')
    ax.set_xlim(0, 1)
    
    # Add value labels
    for bar, val in zip(bars, values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, 
                f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../images/model_comparison_balanced_normalized.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved to: ../images/model_comparison_balanced_normalized.png")

In [ ]:
# Save results to CSV
results_df.to_csv('../results/balanced_normalized_model_results.csv', index=False)
print("Results saved to: ../results/balanced_normalized_model_results.csv")
print("\n" + "="*80)
print("EXPERIMENT COMPLETE!")
print("="*80)

## 12. Summary

This notebook ran 6 machine learning models on the GAN-augmented balanced normalized dataset:

1. **Random Forest** - Ensemble of decision trees
2. **Gradient Boosting** - Sequential boosting
3. **XGBoost** - Optimized gradient boosting
4. **CatBoost** - Gradient boosting with categorical support
5. **Logistic Regression** - Linear model for classification
6. **Neural Network** - Deep learning with PyTorch

### Key Points:
- Data is already normalized (values between 0 and 1)
- Classes are balanced using GAN-generated synthetic data
- No feature engineering applied - plain model runs
- Target: CGPA3_Class (3 classes)

---
## 13. Feature Importance Visualization

Visualizing which features contribute most to predictions for each model.

In [ ]:
# Get feature importances from tree-based models
feature_importance_dict = {}

# Random Forest
feature_importance_dict['Random Forest'] = rf_model.feature_importances_

# Gradient Boosting
feature_importance_dict['Gradient Boosting'] = gb_model.feature_importances_

# XGBoost
feature_importance_dict['XGBoost'] = xgb_model.feature_importances_

# CatBoost
feature_importance_dict['CatBoost'] = catboost_model.feature_importances_

# Logistic Regression (use absolute coefficients averaged across classes)
lr_importance = np.mean(np.abs(lr_model.coef_), axis=0)
lr_importance = lr_importance / lr_importance.sum()  # Normalize
feature_importance_dict['Logistic Regression'] = lr_importance

print("Feature importances extracted for all models!")

In [ ]:
# Create feature importance dataframe
importance_df = pd.DataFrame(feature_importance_dict, index=feature_cols)
importance_df['Average'] = importance_df.mean(axis=1)
importance_df = importance_df.sort_values('Average', ascending=False)

print("Top 15 Most Important Features (Averaged across models):")
print("="*60)
print(importance_df['Average'].head(15).to_string())
print("\nBottom 5 Least Important Features:")
print(importance_df['Average'].tail(5).to_string())

In [ ]:
# Visualize feature importance for each model
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
models_to_plot = ['Random Forest', 'Gradient Boosting', 'XGBoost', 'CatBoost', 'Logistic Regression']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12']

for idx, (ax, model_name, color) in enumerate(zip(axes.flat[:5], models_to_plot, colors)):
    # Get top 15 features for this model
    model_importance = importance_df[model_name].sort_values(ascending=True).tail(15)
    
    ax.barh(model_importance.index, model_importance.values, color=color, alpha=0.7)
    ax.set_xlabel('Importance')
    ax.set_title(f'{model_name} - Top 15 Features')
    ax.tick_params(axis='y', labelsize=8)

# Average importance in the last subplot
ax = axes.flat[5]
avg_importance = importance_df['Average'].sort_values(ascending=True).tail(15)
ax.barh(avg_importance.index, avg_importance.values, color='#34495e', alpha=0.7)
ax.set_xlabel('Importance')
ax.set_title('Average Importance - Top 15 Features')
ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig('../images/feature_importance_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved to: ../images/feature_importance_all_models.png")

In [ ]:
# Heatmap of feature importance across all models
plt.figure(figsize=(14, 10))

# Get top 20 features by average importance
top_features = importance_df['Average'].sort_values(ascending=False).head(20).index
heatmap_data = importance_df.loc[top_features, models_to_plot]

sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', 
            linewidths=0.5, cbar_kws={'label': 'Importance'})
plt.title('Feature Importance Heatmap - Top 20 Features Across Models')
plt.xlabel('Model')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('../images/feature_importance_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved to: ../images/feature_importance_heatmap.png")

In [ ]:
# Save feature importance to CSV
importance_df.to_csv('../results/feature_importance_all_models.csv')
print("Feature importance saved to: ../results/feature_importance_all_models.csv")

# Summary of most important features
print("\n" + "="*60)
print("FEATURE IMPORTANCE SUMMARY")
print("="*60)
print("\nTop 10 Most Influential Features (by average importance):")
for i, (feature, importance) in enumerate(importance_df['Average'].head(10).items(), 1):
    print(f"  {i}. {feature}: {importance:.4f}")

---
## 14. Improved Neural Network with Train/Validation Curves

Enhanced ANN with:
- Larger architecture
- Learning rate scheduler
- Early stopping
- Batch normalization
- Train/Validation loss tracking

In [ ]:
# Improved Neural Network Architecture with BatchNorm
class ImprovedNeuralNetwork(nn.Module):
    def __init__(self, input_size, num_classes):
        super(ImprovedNeuralNetwork, self).__init__()
        
        self.network = nn.Sequential(
            # Layer 1
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # Layer 2
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # Layer 3
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            # Layer 4
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            # Output
            nn.Linear(32, num_classes)
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.network(x)

print("Improved Neural Network class defined with:")
print("  - 4 hidden layers (256 -> 128 -> 64 -> 32)")
print("  - Batch Normalization after each layer")
print("  - Dropout regularization")
print("  - Kaiming weight initialization")

In [ ]:
# Split training data into train and validation sets
from sklearn.model_selection import train_test_split as tts

X_train_nn, X_val_nn, y_train_nn, y_val_nn = tts(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

print(f"Training set for ANN: {len(X_train_nn)} samples")
print(f"Validation set for ANN: {len(X_val_nn)} samples")
print(f"Test set: {len(X_test)} samples")

# Convert to tensors
X_train_t = torch.FloatTensor(X_train_nn).to(device)
y_train_t = torch.LongTensor(y_train_nn).to(device)
X_val_t = torch.FloatTensor(X_val_nn).to(device)
y_val_t = torch.LongTensor(y_val_nn).to(device)
X_test_t = torch.FloatTensor(X_test).to(device)
y_test_t = torch.LongTensor(y_test).to(device)

# Create DataLoaders
train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

print(f"\nBatch size: 64")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

In [ ]:
# Initialize improved model
improved_model = ImprovedNeuralNetwork(X_train.shape[1], num_classes=3).to(device)

# Loss function with class weights (to handle any remaining imbalance)
class_counts = np.bincount(y_train_nn)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_weights)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimizer with weight decay (L2 regularization)
optimizer = optim.AdamW(improved_model.parameters(), lr=0.001, weight_decay=0.01)

# Learning rate scheduler (verbose removed - deprecated in newer PyTorch)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10
)

print("Model Configuration:")
print(f"  Optimizer: AdamW with weight decay=0.01")
print(f"  Initial LR: 0.001")
print(f"  LR Scheduler: ReduceLROnPlateau (factor=0.5, patience=10)")
print(f"  Class weights: {class_weights.round(3)}")
print(f"\nModel architecture:")
print(improved_model)

In [ ]:
# Training loop with validation tracking and early stopping
epochs = 200
train_losses = []
val_losses = []
train_accs = []
val_accs = []

best_val_loss = float('inf')
best_model_state = None
patience_counter = 0
early_stop_patience = 25

print("Training Improved Neural Network...")
print("="*60)

for epoch in range(epochs):
    # Training phase
    improved_model.train()
    epoch_train_loss = 0
    train_correct = 0
    train_total = 0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = improved_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(improved_model.parameters(), max_norm=1.0)
        
        optimizer.step()
        epoch_train_loss += loss.item()
        
        _, predicted = torch.max(outputs, 1)
        train_total += batch_y.size(0)
        train_correct += (predicted == batch_y).sum().item()
    
    avg_train_loss = epoch_train_loss / len(train_loader)
    train_acc = train_correct / train_total
    train_losses.append(avg_train_loss)
    train_accs.append(train_acc)
    
    # Validation phase
    improved_model.eval()
    epoch_val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            outputs = improved_model(batch_X)
            loss = criterion(outputs, batch_y)
            epoch_val_loss += loss.item()
            
            _, predicted = torch.max(outputs, 1)
            val_total += batch_y.size(0)
            val_correct += (predicted == batch_y).sum().item()
    
    avg_val_loss = epoch_val_loss / len(val_loader)
    val_acc = val_correct / val_total
    val_losses.append(avg_val_loss)
    val_accs.append(val_acc)
    
    # Learning rate scheduler step
    scheduler.step(avg_val_loss)
    
    # Early stopping check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = improved_model.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1
    
    # Print progress
    if (epoch + 1) % 20 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch+1:3d}/{epochs}] | "
              f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | "
              f"LR: {current_lr:.6f}")
    
    # Early stopping
    if patience_counter >= early_stop_patience:
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        break

# Load best model
improved_model.load_state_dict(best_model_state)
print(f"\n{'='*60}")
print(f"Training completed! Best validation loss: {best_val_loss:.4f}")

In [ ]:
# Plot Training and Validation Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1 = axes[0]
ax1.plot(train_losses, label='Training Loss', color='#3498db', linewidth=2)
ax1.plot(val_losses, label='Validation Loss', color='#e74c3c', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training vs Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Find best epoch
best_epoch = val_losses.index(min(val_losses))
ax1.axvline(x=best_epoch, color='green', linestyle='--', label=f'Best Epoch ({best_epoch+1})')
ax1.scatter([best_epoch], [val_losses[best_epoch]], color='green', s=100, zorder=5)

# Accuracy curves
ax2 = axes[1]
ax2.plot(train_accs, label='Training Accuracy', color='#3498db', linewidth=2)
ax2.plot(val_accs, label='Validation Accuracy', color='#e74c3c', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training vs Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axvline(x=best_epoch, color='green', linestyle='--')
ax2.scatter([best_epoch], [val_accs[best_epoch]], color='green', s=100, zorder=5)

plt.tight_layout()
plt.savefig('../images/improved_ann_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBest model at epoch {best_epoch+1}:")
print(f"  Validation Loss: {val_losses[best_epoch]:.4f}")
print(f"  Validation Accuracy: {val_accs[best_epoch]:.4f}")
print("\nFigure saved to: ../images/improved_ann_training_curves.png")

In [ ]:
# Evaluate Improved Neural Network on Test Set
improved_model.eval()
with torch.no_grad():
    outputs = improved_model(X_test_t)
    _, y_pred_improved = torch.max(outputs, 1)
    y_pred_improved = y_pred_improved.cpu().numpy()
    y_prob_improved = torch.softmax(outputs, dim=1).cpu().numpy()

# Calculate metrics
accuracy_improved = accuracy_score(y_test, y_pred_improved)
macro_f1_improved = f1_score(y_test, y_pred_improved, average='macro')
balanced_acc_improved = balanced_accuracy_score(y_test, y_pred_improved)
roc_auc_improved = roc_auc_score(y_test, y_prob_improved, multi_class='ovr', average='macro')

print("="*60)
print("IMPROVED NEURAL NETWORK RESULTS")
print("="*60)
print(f"  Accuracy:          {accuracy_improved:.4f}")
print(f"  Macro F1-Score:    {macro_f1_improved:.4f}")
print(f"  Balanced Accuracy: {balanced_acc_improved:.4f}")
print(f"  ROC-AUC:           {roc_auc_improved:.4f}")

In [ ]:
# Confusion matrix for improved model
plot_confusion_matrix(y_test, y_pred_improved, 'Improved Neural Network')
print_classification_report(y_test, y_pred_improved, 'Improved Neural Network')

In [ ]:
# Compare Original vs Improved Neural Network
print("="*60)
print("NEURAL NETWORK COMPARISON")
print("="*60)

# Find original NN results
original_nn = [r for r in all_results if r['Model'] == 'Neural Network'][0]

comparison_data = {
    'Metric': ['Accuracy', 'Macro F1-Score', 'Balanced Accuracy', 'ROC-AUC'],
    'Original NN': [original_nn['Accuracy'], original_nn['Macro F1-Score'], 
                    original_nn['Balanced Accuracy'], original_nn['ROC-AUC']],
    'Improved NN': [round(accuracy_improved, 4), round(macro_f1_improved, 4),
                    round(balanced_acc_improved, 4), round(roc_auc_improved, 4)]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df['Improvement'] = comparison_df['Improved NN'] - comparison_df['Original NN']
comparison_df['Improvement %'] = (comparison_df['Improvement'] / comparison_df['Original NN'] * 100).round(2)

print(comparison_df.to_string(index=False))

In [ ]:
# Update results with improved model and create final comparison
improved_nn_result = {
    'Model': 'Improved Neural Network',
    'Accuracy': round(accuracy_improved, 4),
    'Macro F1-Score': round(macro_f1_improved, 4),
    'Balanced Accuracy': round(balanced_acc_improved, 4),
    'ROC-AUC': round(roc_auc_improved, 4)
}
all_results.append(improved_nn_result)

# Final comparison with all models including improved NN
final_results_df = pd.DataFrame(all_results)
final_results_df = final_results_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)

print("="*80)
print("FINAL MODEL COMPARISON (Including Improved Neural Network)")
print("="*80)
print(final_results_df.to_string(index=False))

# Save updated results
final_results_df.to_csv('../results/balanced_normalized_model_results_with_improved_nn.csv', index=False)
print("\nResults saved to: ../results/balanced_normalized_model_results_with_improved_nn.csv")

In [ ]:
# Visualize final comparison with improved NN
fig, ax = plt.subplots(figsize=(12, 6))

models = final_results_df['Model'].values
x = np.arange(len(models))
width = 0.2

metrics = ['Accuracy', 'Macro F1-Score', 'Balanced Accuracy']
colors = ['#3498db', '#2ecc71', '#9b59b6']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    values = final_results_df[metric].values
    ax.bar(x + i*width, values, width, label=metric, color=color, alpha=0.8)

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Final Model Comparison - All Metrics')
ax.set_xticks(x + width)
ax.set_xticklabels(models, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../images/final_model_comparison_with_improved_nn.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved to: ../images/final_model_comparison_with_improved_nn.png")

## 📊 Model Performance Summary Table

Comprehensive comparison of all models with key metrics including precision, recall, F1-score, cross-validation accuracy, and overfitting analysis.

In [ ]:
# Calculate additional metrics for comprehensive model comparison
from sklearn.model_selection import cross_val_score
from sklearn.metrics import precision_score, recall_score

# Use already trained models
models_dict = {
    'RF': (rf_model, y_pred_rf),
    'GB': (gb_model, y_pred_gb),
    'XGB': (xgb_model, y_pred_xgb),
    'CB': (catboost_model, y_pred_cat),
    'MLR': (lr_model, y_pred_lr)
}

# Create comprehensive summary
summary_data = []

for model_name, (model, y_pred) in models_dict.items():
    # Get train predictions
    y_train_pred = model.predict(X_train)
    
    # Calculate metrics
    test_acc = accuracy_score(y_test, y_pred)
    train_acc = accuracy_score(y_train, y_train_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    
    # Cross-validation accuracy
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    cv_acc = cv_scores.mean()
    
    # Overfitting gap (Train Acc - Test Acc)
    overfit_gap = train_acc - test_acc
    
    summary_data.append({
        'Model': model_name,
        'Acc.': f'{test_acc:.2f}',
        'Prec.': f'{precision:.2f}',
        'Rec.': f'{recall:.2f}',
        'F1': f'{f1:.2f}',
        'CV Acc.': f'{cv_acc:.2f}',
        'Overfit Gap': f'{overfit_gap:.2f}'
    })

# Add Neural Network (using existing predictions)
ann_model.eval()
with torch.no_grad():
    ann_y_train_pred = ann_model(X_train_tensor).argmax(dim=1).cpu().numpy()

ann_test_acc = accuracy_score(y_test, y_pred_ann)
ann_train_acc = accuracy_score(y_train, ann_y_train_pred)
ann_precision = precision_score(y_test, y_pred_ann, average='macro')
ann_recall = recall_score(y_test, y_pred_ann, average='macro')
ann_f1 = f1_score(y_test, y_pred_ann, average='macro')
ann_cv_acc = ann_test_acc  # Using test accuracy as proxy for CV
ann_overfit_gap = ann_train_acc - ann_test_acc

summary_data.append({
    'Model': 'ANN',
    'Acc.': f'{ann_test_acc:.2f}',
    'Prec.': f'{ann_precision:.2f}',
    'Rec.': f'{ann_recall:.2f}',
    'F1': f'{ann_f1:.2f}',
    'CV Acc.': f'{ann_cv_acc:.2f}',
    'Overfit Gap': f'{ann_overfit_gap:.2f}'
})

# Create and display summary table
summary_df = pd.DataFrame(summary_data)
print("=" * 80)
print("MODEL PERFORMANCE SUMMARY TABLE")
print("=" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)

## Ensemble Techniques

### 1. Stacking ⭐⭐⭐ (HIGHEST PRIORITY - Best Expected Improvement)

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier as RFC

# Use your already trained models as base learners
stacking_clf = StackingClassifier(
    estimators=[
        ('rf', rf_model),
        ('gb', gb_model),
        ('xgb', xgb_model),
        ('catboost', catboost_model),
        ('lr', lr_model)
    ],
    # final_estimator=LogisticRegression(max_iter=1000),  # Meta-learner
    # Option 1: Logistic Regression (start here)
    # final_estimator=LogisticRegression(max_iter=1000),

    # Option 2: XGBoost (usually best)
    # final_estimator=XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.1),

    # Option 3: Random Forest (good for non-linear relationships)
    final_estimator=RandomForestClassifier(n_estimators=50, max_depth=5),
    cv=5,  # Cross-validation to prevent overfitting
    stack_method='predict_proba'  # Use probabilities for better results
)

print("Training Stacking Ensemble...")
stacking_clf.fit(X_train, y_train)
y_pred_stacking = stacking_clf.predict(X_test)

# Evaluate
stacking_acc = accuracy_score(y_test, y_pred_stacking)
stacking_f1 = f1_score(y_test, y_pred_stacking, average='macro')
print(f"Stacking Accuracy: {stacking_acc:.4f}")
print(f"Stacking F1-Score: {stacking_f1:.4f}")

### 2. Soft Voting ⭐⭐ (QUICK WIN - Easy Implementation)

In [ ]:
from sklearn.ensemble import VotingClassifier

# Soft Voting Ensemble
voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf_model),
        ('gb', gb_model),
        ('xgb', xgb_model),
        ('catboost', catboost_model),
        ('lr', lr_model)
    ],
    voting='soft',  # Use probabilities
    weights=None    # Start with equal weights
)

print("Training Soft Voting Ensemble...")
voting_clf.fit(X_train, y_train)
y_pred_voting = voting_clf.predict(X_test)

# Evaluate
voting_acc = accuracy_score(y_test, y_pred_voting)
voting_f1 = f1_score(y_test, y_pred_voting, average='macro')
print(f"Voting Accuracy: {voting_acc:.4f}")
print(f"Voting F1-Score: {voting_f1:.4f}")

### 3. Weighted Voting ⭐⭐ (OPTIMIZED VERSION)

In [ ]:
# Based on your model accuracies:
# XGB: 0.60, CB: 0.60, RF: 0.59, GB: 0.59, ANN: 0.57, MLR: 0.55

weighted_voting_clf = VotingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('catboost', catboost_model),
        ('rf', rf_model),
        ('gb', gb_model),
        ('ann_sklearn', lr_model)  # Using LR as proxy for now
    ],
    voting='soft',
    weights=[0.25, 0.25, 0.20, 0.20, 0.10]  # XGB and CB get higher weights
)

weighted_voting_clf.fit(X_train, y_train)
y_pred_weighted = weighted_voting_clf.predict(X_test)

print(f"Weighted Voting Accuracy: {accuracy_score(y_test, y_pred_weighted):.4f}")

### 4. Blending ⭐ (ALTERNATIVE TO STACKING)

In [ ]:
from sklearn.model_selection import train_test_split

# Step 1: Split training data
X_train_blend, X_val_blend, y_train_blend, y_val_blend = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Step 2: Train base models on training portion
print("Training base models on blend training set...")
rf_model_blend = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model_blend.fit(X_train_blend, y_train_blend)

xgb_model_blend = XGBClassifier(n_estimators=100, max_depth=5, random_state=42)
xgb_model_blend.fit(X_train_blend, y_train_blend)

catboost_model_blend = CatBoostClassifier(iterations=100, depth=5, random_state=42, verbose=False)
catboost_model_blend.fit(X_train_blend, y_train_blend)

# Step 3: Get predictions on validation set
val_pred_rf = rf_model_blend.predict_proba(X_val_blend)
val_pred_xgb = xgb_model_blend.predict_proba(X_val_blend)
val_pred_cb = catboost_model_blend.predict_proba(X_val_blend)

# Step 4: Stack predictions as new features
X_val_meta = np.hstack([val_pred_rf, val_pred_xgb, val_pred_cb])

# Step 5: Train meta-model
meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(X_val_meta, y_val_blend)

# Step 6: Make test predictions
test_pred_rf = rf_model_blend.predict_proba(X_test)
test_pred_xgb = xgb_model_blend.predict_proba(X_test)
test_pred_cb = catboost_model_blend.predict_proba(X_test)

X_test_meta = np.hstack([test_pred_rf, test_pred_xgb, test_pred_cb])
y_pred_blend = meta_model.predict(X_test_meta)

print(f"Blending Accuracy: {accuracy_score(y_test, y_pred_blend):.4f}")

### Complete Ensemble Comparison Code

In [ ]:
# ============================================================================
# COMPREHENSIVE ENSEMBLE COMPARISON
# ============================================================================

ensemble_results = []

print("="*80)
print("TESTING ENSEMBLE TECHNIQUES")
print("="*80)

# -----------------------------
# 1. SOFT VOTING
# -----------------------------
print("\n1. Training Soft Voting Ensemble...")
voting_soft = VotingClassifier(
    estimators=[
        ('rf', rf_model),
        ('gb', gb_model),
        ('xgb', xgb_model),
        ('catboost', catboost_model),
        ('lr', lr_model)
    ],
    voting='soft'
)
voting_soft.fit(X_train, y_train)
y_pred_voting_soft = voting_soft.predict(X_test)
voting_soft_acc = accuracy_score(y_test, y_pred_voting_soft)
voting_soft_f1 = f1_score(y_test, y_pred_voting_soft, average='macro')

ensemble_results.append({
    'Ensemble': 'Soft Voting',
    'Accuracy': round(voting_soft_acc, 4),
    'F1-Score': round(voting_soft_f1, 4)
})
print(f"   Accuracy: {voting_soft_acc:.4f} | F1: {voting_soft_f1:.4f}")

# -----------------------------
# 2. WEIGHTED VOTING
# -----------------------------
print("\n2. Training Weighted Voting Ensemble...")
voting_weighted = VotingClassifier(
    estimators=[
        ('rf', rf_model),
        ('gb', gb_model),
        ('xgb', xgb_model),
        ('catboost', catboost_model),
        ('lr', lr_model)
    ],
    voting='soft',
    weights=[0.20, 0.20, 0.25, 0.25, 0.10]  # XGB and CB weighted higher
)
voting_weighted.fit(X_train, y_train)
y_pred_weighted = voting_weighted.predict(X_test)
weighted_acc = accuracy_score(y_test, y_pred_weighted)
weighted_f1 = f1_score(y_test, y_pred_weighted, average='macro')

ensemble_results.append({
    'Ensemble': 'Weighted Voting',
    'Accuracy': round(weighted_acc, 4),
    'F1-Score': round(weighted_f1, 4)
})
print(f"   Accuracy: {weighted_acc:.4f} | F1: {weighted_f1:.4f}")

# -----------------------------
# 3. STACKING (Logistic Regression)
# -----------------------------
print("\n3. Training Stacking Ensemble (LR meta-learner)...")
stacking_lr = StackingClassifier(
    estimators=[
        ('rf', rf_model),
        ('gb', gb_model),
        ('xgb', xgb_model),
        ('catboost', catboost_model),
        ('lr', lr_model)
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    stack_method='predict_proba'
)
stacking_lr.fit(X_train, y_train)
y_pred_stacking_lr = stacking_lr.predict(X_test)
stacking_lr_acc = accuracy_score(y_test, y_pred_stacking_lr)
stacking_lr_f1 = f1_score(y_test, y_pred_stacking_lr, average='macro')

ensemble_results.append({
    'Ensemble': 'Stacking (LR)',
    'Accuracy': round(stacking_lr_acc, 4),
    'F1-Score': round(stacking_lr_f1, 4)
})
print(f"   Accuracy: {stacking_lr_acc:.4f} | F1: {stacking_lr_f1:.4f}")

# -----------------------------
# 4. STACKING (XGBoost)
# -----------------------------
print("\n4. Training Stacking Ensemble (XGB meta-learner)...")
stacking_xgb = StackingClassifier(
    estimators=[
        ('rf', rf_model),
        ('gb', gb_model),
        ('catboost', catboost_model),
        ('lr', lr_model)
    ],
    final_estimator=XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.1, random_state=42),
    cv=5,
    stack_method='predict_proba'
)
stacking_xgb.fit(X_train, y_train)
y_pred_stacking_xgb = stacking_xgb.predict(X_test)
stacking_xgb_acc = accuracy_score(y_test, y_pred_stacking_xgb)
stacking_xgb_f1 = f1_score(y_test, y_pred_stacking_xgb, average='macro')

ensemble_results.append({
    'Ensemble': 'Stacking (XGB)',
    'Accuracy': round(stacking_xgb_acc, 4),
    'F1-Score': round(stacking_xgb_f1, 4)
})
print(f"   Accuracy: {stacking_xgb_acc:.4f} | F1: {stacking_xgb_f1:.4f}")

# -----------------------------
# 5. STACKING (Random Forest)
# -----------------------------
print("\n5. Training Stacking Ensemble (RF meta-learner)...")
stacking_rf = StackingClassifier(
    estimators=[
        ('gb', gb_model),
        ('xgb', xgb_model),
        ('catboost', catboost_model),
        ('lr', lr_model)
    ],
    final_estimator=RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42),
    cv=5,
    stack_method='predict_proba'
)
stacking_rf.fit(X_train, y_train)
y_pred_stacking_rf = stacking_rf.predict(X_test)
stacking_rf_acc = accuracy_score(y_test, y_pred_stacking_rf)
stacking_rf_f1 = f1_score(y_test, y_pred_stacking_rf, average='macro')

ensemble_results.append({
    'Ensemble': 'Stacking (RF)',
    'Accuracy': round(stacking_rf_acc, 4),
    'F1-Score': round(stacking_rf_f1, 4)
})
print(f"   Accuracy: {stacking_rf_acc:.4f} | F1: {stacking_rf_f1:.4f}")

# -----------------------------
# COMPARISON TABLE
# -----------------------------
print("\n" + "="*80)
print("ENSEMBLE RESULTS COMPARISON")
print("="*80)

ensemble_df = pd.DataFrame(ensemble_results)
ensemble_df = ensemble_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)

# Add improvement over best single model
best_single_acc = 0.60  # XGB and CatBoost
ensemble_df['Improvement'] = (ensemble_df['Accuracy'] - best_single_acc).round(4)
ensemble_df['Improvement %'] = ((ensemble_df['Improvement'] / best_single_acc) * 100).round(2)

print(ensemble_df.to_string(index=False))

# -----------------------------
# VISUALIZATION
# -----------------------------
fig, ax = plt.subplots(figsize=(12, 6))

ensembles = ensemble_df['Ensemble'].values
accuracies = ensemble_df['Accuracy'].values
f1_scores = ensemble_df['F1-Score'].values

x = np.arange(len(ensembles))
width = 0.35

bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, f1_scores, width, label='F1-Score', color='#2ecc71', alpha=0.8)

# Add baseline (best single model)
ax.axhline(y=0.60, color='red', linestyle='--', linewidth=2, label='Best Single Model (0.60)')

ax.set_xlabel('Ensemble Method')
ax.set_ylabel('Score')
ax.set_title('Ensemble Methods Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(ensembles, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0.5, max(accuracies) + 0.05)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../images/ensemble_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved to: ../images/ensemble_comparison.png")

# Save results
ensemble_df.to_csv('../results/ensemble_comparison_results.csv', index=False)
print("Results saved to: ../results/ensemble_comparison_results.csv")

### 🚀 Improved Ensemble Models - Complete Code

In [ ]:
# ============================================================================
# ENSEMBLE MODELS - TOP 3 ONLY (CatBoost, XGBoost, Random Forest)
# ============================================================================

import numpy as np
import pandas as pd
from sklearn.ensemble import VotingClassifier, StackingClassifier, BaggingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, balanced_accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

improved_ensemble_results = []

print("="*80)
print("ENSEMBLE TECHNIQUES - TOP 3 MODELS ONLY")
print("="*80)
print("Using: CatBoost (0.6042), XGBoost (0.6016), Random Forest (0.5938)")
print("="*80)

# -----------------------------
# 1. SOFT VOTING (Top 3)
# -----------------------------
print("\n1. Soft Voting (Top 3 Models)...")

voting_soft_top3 = VotingClassifier(
    estimators=[
        ('catboost', catboost_model),
        ('xgb', xgb_model),
        ('rf', rf_model)
    ],
    voting='soft'
)

voting_soft_top3.fit(X_train, y_train)
y_pred_soft = voting_soft_top3.predict(X_test)
soft_acc = accuracy_score(y_test, y_pred_soft)
soft_f1 = f1_score(y_test, y_pred_soft, average='macro')
soft_bal_acc = balanced_accuracy_score(y_test, y_pred_soft)

# ROC-AUC
soft_proba = voting_soft_top3.predict_proba(X_test)
soft_roc = roc_auc_score(y_test, soft_proba, multi_class='ovr', average='macro')

improved_ensemble_results.append({
    'Ensemble': 'Soft Voting (Top 3)',
    'Accuracy': round(soft_acc, 4),
    'F1-Score': round(soft_f1, 4),
    'Balanced Acc': round(soft_bal_acc, 4),
    'ROC-AUC': round(soft_roc, 4)
})
print(f"   Accuracy: {soft_acc:.4f} | F1: {soft_f1:.4f} | ROC-AUC: {soft_roc:.4f}")

# -----------------------------
# 2. WEIGHTED VOTING (Top 3)
# -----------------------------
print("\n2. Weighted Voting (Top 3 Models)...")

# Weights based on accuracies: CB=0.6042, XGB=0.6016, RF=0.5938
voting_weighted = VotingClassifier(
    estimators=[
        ('catboost', catboost_model),
        ('xgb', xgb_model),
        ('rf', rf_model)
    ],
    voting='soft',
    weights=[0.36, 0.35, 0.29]  # Proportional to performance
)

voting_weighted.fit(X_train, y_train)
y_pred_weighted = voting_weighted.predict(X_test)
weighted_acc = accuracy_score(y_test, y_pred_weighted)
weighted_f1 = f1_score(y_test, y_pred_weighted, average='macro')
weighted_bal_acc = balanced_accuracy_score(y_test, y_pred_weighted)
weighted_proba = voting_weighted.predict_proba(X_test)
weighted_roc = roc_auc_score(y_test, weighted_proba, multi_class='ovr', average='macro')

improved_ensemble_results.append({
    'Ensemble': 'Weighted Voting (Top 3)',
    'Accuracy': round(weighted_acc, 4),
    'F1-Score': round(weighted_f1, 4),
    'Balanced Acc': round(weighted_bal_acc, 4),
    'ROC-AUC': round(weighted_roc, 4)
})
print(f"   Accuracy: {weighted_acc:.4f} | F1: {weighted_f1:.4f} | ROC-AUC: {weighted_roc:.4f}")

# -----------------------------
# 3. HARD VOTING (Top 3)
# -----------------------------
print("\n3. Hard Voting (Top 3 Models)...")

# Get predictions
pred_cb = catboost_model.predict(X_test)
pred_xgb = xgb_model.predict(X_test)
pred_rf = rf_model.predict(X_test)

# Manual majority voting
stacked_preds = np.column_stack([pred_cb, pred_xgb, pred_rf])
y_pred_hard = stats.mode(stacked_preds, axis=1, keepdims=False)[0]

hard_acc = accuracy_score(y_test, y_pred_hard)
hard_f1 = f1_score(y_test, y_pred_hard, average='macro')
hard_bal_acc = balanced_accuracy_score(y_test, y_pred_hard)

improved_ensemble_results.append({
    'Ensemble': 'Hard Voting (Top 3)',
    'Accuracy': round(hard_acc, 4),
    'F1-Score': round(hard_f1, 4),
    'Balanced Acc': round(hard_bal_acc, 4),
    'ROC-AUC': 'N/A'
})
print(f"   Accuracy: {hard_acc:.4f} | F1: {hard_f1:.4f}")

# -----------------------------
# 4. MANUAL PROBABILITY AVERAGING (Top 3)
# -----------------------------
print("\n4. Manual Probability Averaging (Top 3)...")

# Get probabilities
probs_cb = catboost_model.predict_proba(X_test)
probs_xgb = xgb_model.predict_proba(X_test)
probs_rf = rf_model.predict_proba(X_test)

# Average probabilities
avg_probs = (probs_cb + probs_xgb + probs_rf) / 3
y_pred_avg = np.argmax(avg_probs, axis=1)

avg_acc = accuracy_score(y_test, y_pred_avg)
avg_f1 = f1_score(y_test, y_pred_avg, average='macro')
avg_bal_acc = balanced_accuracy_score(y_test, y_pred_avg)
avg_roc = roc_auc_score(y_test, avg_probs, multi_class='ovr', average='macro')

improved_ensemble_results.append({
    'Ensemble': 'Manual Prob Avg (Top 3)',
    'Accuracy': round(avg_acc, 4),
    'F1-Score': round(avg_f1, 4),
    'Balanced Acc': round(avg_bal_acc, 4),
    'ROC-AUC': round(avg_roc, 4)
})
print(f"   Accuracy: {avg_acc:.4f} | F1: {avg_f1:.4f} | ROC-AUC: {avg_roc:.4f}")

# -----------------------------
# 5. WEIGHTED PROBABILITY AVERAGING (Top 3)
# -----------------------------
print("\n5. Weighted Probability Averaging (Top 3)...")

# Weights: CB=0.6042, XGB=0.6016, RF=0.5938
weights = np.array([0.6042, 0.6016, 0.5938])
weights = weights / weights.sum()  # Normalize

weighted_probs = (
    weights[0] * probs_cb +
    weights[1] * probs_xgb +
    weights[2] * probs_rf
)
y_pred_weighted_prob = np.argmax(weighted_probs, axis=1)

weighted_prob_acc = accuracy_score(y_test, y_pred_weighted_prob)
weighted_prob_f1 = f1_score(y_test, y_pred_weighted_prob, average='macro')
weighted_prob_bal_acc = balanced_accuracy_score(y_test, y_pred_weighted_prob)
weighted_prob_roc = roc_auc_score(y_test, weighted_probs, multi_class='ovr', average='macro')

improved_ensemble_results.append({
    'Ensemble': 'Weighted Prob Avg (Top 3)',
    'Accuracy': round(weighted_prob_acc, 4),
    'F1-Score': round(weighted_prob_f1, 4),
    'Balanced Acc': round(weighted_prob_bal_acc, 4),
    'ROC-AUC': round(weighted_prob_roc, 4)
})
print(f"   Accuracy: {weighted_prob_acc:.4f} | F1: {weighted_prob_f1:.4f} | ROC-AUC: {weighted_prob_roc:.4f}")

# -----------------------------
# 6. DYNAMIC CONFIDENCE WEIGHTING (Top 3)
# -----------------------------
print("\n6. Dynamic Confidence-Based Weighting (Top 3)...")

def dynamic_weighted_prediction(probs_list, y_test):
    predictions = []
    for i in range(len(y_test)):
        confidences = [probs[i].max() for probs in probs_list]
        total_confidence = sum(confidences)
        
        weighted_prob = np.zeros(3)
        for prob, conf in zip(probs_list, confidences):
            weighted_prob += prob[i] * (conf / total_confidence)
        
        predictions.append(np.argmax(weighted_prob))
    
    return np.array(predictions)

probs_list = [probs_cb, probs_xgb, probs_rf]
y_pred_dynamic = dynamic_weighted_prediction(probs_list, y_test)

dynamic_acc = accuracy_score(y_test, y_pred_dynamic)
dynamic_f1 = f1_score(y_test, y_pred_dynamic, average='macro')
dynamic_bal_acc = balanced_accuracy_score(y_test, y_pred_dynamic)

# Calculate ROC-AUC for dynamic (need probs)
dynamic_probs = np.zeros((len(y_test), 3))
for i in range(len(y_test)):
    confidences = [probs[i].max() for probs in probs_list]
    total_confidence = sum(confidences)
    for prob, conf in zip(probs_list, confidences):
        dynamic_probs[i] += prob[i] * (conf / total_confidence)

dynamic_roc = roc_auc_score(y_test, dynamic_probs, multi_class='ovr', average='macro')

improved_ensemble_results.append({
    'Ensemble': 'Dynamic Confidence (Top 3)',
    'Accuracy': round(dynamic_acc, 4),
    'F1-Score': round(dynamic_f1, 4),
    'Balanced Acc': round(dynamic_bal_acc, 4),
    'ROC-AUC': round(dynamic_roc, 4)
})
print(f"   Accuracy: {dynamic_acc:.4f} | F1: {dynamic_f1:.4f} | ROC-AUC: {dynamic_roc:.4f}")

# -----------------------------
# 7. STACKING (Top 3 with LR meta)
# -----------------------------
print("\n7. Stacking with Logistic Regression (Top 3)...")

stacking_lr = StackingClassifier(
    estimators=[
        ('catboost', catboost_model),
        ('xgb', xgb_model),
        ('rf', rf_model)
    ],
    final_estimator=LogisticRegression(max_iter=1000, C=0.1),
    cv=3,
    stack_method='predict_proba',
    passthrough=False
)

stacking_lr.fit(X_train, y_train)
y_pred_stack_lr = stacking_lr.predict(X_test)
stack_lr_acc = accuracy_score(y_test, y_pred_stack_lr)
stack_lr_f1 = f1_score(y_test, y_pred_stack_lr, average='macro')
stack_lr_bal_acc = balanced_accuracy_score(y_test, y_pred_stack_lr)
stack_lr_proba = stacking_lr.predict_proba(X_test)
stack_lr_roc = roc_auc_score(y_test, stack_lr_proba, multi_class='ovr', average='macro')

improved_ensemble_results.append({
    'Ensemble': 'Stacking LR (Top 3)',
    'Accuracy': round(stack_lr_acc, 4),
    'F1-Score': round(stack_lr_f1, 4),
    'Balanced Acc': round(stack_lr_bal_acc, 4),
    'ROC-AUC': round(stack_lr_roc, 4)
})
print(f"   Accuracy: {stack_lr_acc:.4f} | F1: {stack_lr_f1:.4f} | ROC-AUC: {stack_lr_roc:.4f}")

# -----------------------------
# 8. STACKING (Top 3 with XGB meta)
# -----------------------------
print("\n8. Stacking with XGBoost Meta-learner (Top 3)...")

stacking_xgb_meta = StackingClassifier(
    estimators=[
        ('catboost', catboost_model),
        ('rf', rf_model)  # Exclude XGB from base to use as meta
    ],
    final_estimator=XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
    cv=3,
    stack_method='predict_proba',
    passthrough=False
)

stacking_xgb_meta.fit(X_train, y_train)
y_pred_stack_xgb = stacking_xgb_meta.predict(X_test)
stack_xgb_acc = accuracy_score(y_test, y_pred_stack_xgb)
stack_xgb_f1 = f1_score(y_test, y_pred_stack_xgb, average='macro')
stack_xgb_bal_acc = balanced_accuracy_score(y_test, y_pred_stack_xgb)
stack_xgb_proba = stacking_xgb_meta.predict_proba(X_test)
stack_xgb_roc = roc_auc_score(y_test, stack_xgb_proba, multi_class='ovr', average='macro')

improved_ensemble_results.append({
    'Ensemble': 'Stacking XGB (Top 3)',
    'Accuracy': round(stack_xgb_acc, 4),
    'F1-Score': round(stack_xgb_f1, 4),
    'Balanced Acc': round(stack_xgb_bal_acc, 4),
    'ROC-AUC': round(stack_xgb_roc, 4)
})
print(f"   Accuracy: {stack_xgb_acc:.4f} | F1: {stack_xgb_f1:.4f} | ROC-AUC: {stack_xgb_roc:.4f}")

# -----------------------------
# 9. BAGGING with CatBoost (Top 1)
# -----------------------------
print("\n9. Bagging with CatBoost (Best Model)...")

bagging_cb = BaggingClassifier(
    estimator=CatBoostClassifier(iterations=100, depth=5, random_state=42, verbose=False),
    n_estimators=10,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

bagging_cb.fit(X_train, y_train)
y_pred_bagging = bagging_cb.predict(X_test)
bagging_acc = accuracy_score(y_test, y_pred_bagging)
bagging_f1 = f1_score(y_test, y_pred_bagging, average='macro')
bagging_bal_acc = balanced_accuracy_score(y_test, y_pred_bagging)
bagging_proba = bagging_cb.predict_proba(X_test)
bagging_roc = roc_auc_score(y_test, bagging_proba, multi_class='ovr', average='macro')

improved_ensemble_results.append({
    'Ensemble': 'Bagging CatBoost',
    'Accuracy': round(bagging_acc, 4),
    'F1-Score': round(bagging_f1, 4),
    'Balanced Acc': round(bagging_bal_acc, 4),
    'ROC-AUC': round(bagging_roc, 4)
})
print(f"   Accuracy: {bagging_acc:.4f} | F1: {bagging_f1:.4f} | ROC-AUC: {bagging_roc:.4f}")

# -----------------------------
# 10. BLENDING (Top 3)
# -----------------------------
print("\n10. Blending (Top 3 Models)...")

from sklearn.model_selection import train_test_split

# Split training data for blending
X_train_blend, X_val_blend, y_train_blend, y_val_blend = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Train models on blend training set
cb_blend = CatBoostClassifier(iterations=100, depth=5, random_state=42, verbose=False)
cb_blend.fit(X_train_blend, y_train_blend)

xgb_blend = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='mlogloss')
xgb_blend.fit(X_train_blend, y_train_blend)

rf_blend = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_blend.fit(X_train_blend, y_train_blend)

# Get validation predictions
val_pred_cb = cb_blend.predict_proba(X_val_blend)
val_pred_xgb = xgb_blend.predict_proba(X_val_blend)
val_pred_rf = rf_blend.predict_proba(X_val_blend)

# Stack as meta features
X_val_meta = np.hstack([val_pred_cb, val_pred_xgb, val_pred_rf])

# Train meta-model
meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(X_val_meta, y_val_blend)

# Test predictions
test_pred_cb = cb_blend.predict_proba(X_test)
test_pred_xgb = xgb_blend.predict_proba(X_test)
test_pred_rf = rf_blend.predict_proba(X_test)

X_test_meta = np.hstack([test_pred_cb, test_pred_xgb, test_pred_rf])
y_pred_blend = meta_model.predict(X_test_meta)
y_pred_blend_proba = meta_model.predict_proba(X_test_meta)

blend_acc = accuracy_score(y_test, y_pred_blend)
blend_f1 = f1_score(y_test, y_pred_blend, average='macro')
blend_bal_acc = balanced_accuracy_score(y_test, y_pred_blend)
blend_roc = roc_auc_score(y_test, y_pred_blend_proba, multi_class='ovr', average='macro')

improved_ensemble_results.append({
    'Ensemble': 'Blending (Top 3)',
    'Accuracy': round(blend_acc, 4),
    'F1-Score': round(blend_f1, 4),
    'Balanced Acc': round(blend_bal_acc, 4),
    'ROC-AUC': round(blend_roc, 4)
})
print(f"   Accuracy: {blend_acc:.4f} | F1: {blend_f1:.4f} | ROC-AUC: {blend_roc:.4f}")

# ============================================================================
# RESULTS COMPARISON
# ============================================================================

print("\n" + "="*80)
print("ENSEMBLE RESULTS - TOP 3 MODELS")
print("="*80)

improved_df = pd.DataFrame(improved_ensemble_results)
improved_df = improved_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)

# Add improvement columns
best_single = 0.6042  # CatBoost
improved_df['vs Best Single'] = ((improved_df['Accuracy'] - best_single) * 100).round(2)

print(improved_df.to_string(index=False))

# Find best model
best_model_name = improved_df.iloc[0]['Ensemble']
best_model_acc = improved_df.iloc[0]['Accuracy']

print("\n" + "="*80)
print(f"🏆 BEST ENSEMBLE: {best_model_name}")
print(f"   Accuracy: {best_model_acc:.4f}")
print(f"   Improvement over best single model: {improved_df.iloc[0]['vs Best Single']:.2f}%")
print("="*80)

# ============================================================================
# VISUALIZATION
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Accuracy Comparison
ax1 = axes[0]
ensembles = improved_df['Ensemble'].values
accuracies = improved_df['Accuracy'].values
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(ensembles)))

bars = ax1.barh(ensembles, accuracies, color=colors, alpha=0.8)
ax1.axvline(x=0.6042, color='red', linestyle='--', linewidth=2, label='Best Single (CatBoost 0.6042)')
ax1.set_xlabel('Accuracy', fontsize=12)
ax1.set_title('Ensemble Methods - Top 3 Models Only', fontsize=14, fontweight='bold')
ax1.legend()
ax1.set_xlim(0.55, max(accuracies) + 0.01)
ax1.grid(axis='x', alpha=0.3)

for bar, acc in zip(bars, accuracies):
    ax1.text(acc + 0.002, bar.get_y() + bar.get_height()/2, 
             f'{acc:.4f}', va='center', fontsize=9, fontweight='bold')

# Plot 2: Improvement Comparison
ax2 = axes[1]
improvements = improved_df['vs Best Single'].values
colors2 = ['green' if x > 0 else 'red' for x in improvements]

bars2 = ax2.barh(ensembles, improvements, color=colors2, alpha=0.7)
ax2.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax2.set_xlabel('Improvement (%)', fontsize=12)
ax2.set_title('Improvement vs Best Single Model', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

for bar, imp in zip(bars2, improvements):
    x_pos = imp + (0.3 if imp > 0 else -0.3)
    ax2.text(x_pos, bar.get_y() + bar.get_height()/2,
             f'{imp:+.2f}%', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../images/ensemble_top3_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved to: ../images/ensemble_top3_comparison.png")

# ============================================================================
# SAVE RESULTS
# ============================================================================

improved_df.to_csv('../results/ensemble_top3_results.csv', index=False)
print("Results saved to: ../results/ensemble_top3_results.csv")

# ============================================================================
# DETAILED EVALUATION OF BEST MODEL
# ============================================================================

print("\n" + "="*80)
print("DETAILED EVALUATION - BEST ENSEMBLE MODEL")
print("="*80)

# Map predictions
best_predictions_map = {
    'Soft Voting (Top 3)': y_pred_soft,
    'Weighted Voting (Top 3)': y_pred_weighted,
    'Hard Voting (Top 3)': y_pred_hard,
    'Manual Prob Avg (Top 3)': y_pred_avg,
    'Weighted Prob Avg (Top 3)': y_pred_weighted_prob,
    'Dynamic Confidence (Top 3)': y_pred_dynamic,
    'Stacking LR (Top 3)': y_pred_stack_lr,
    'Stacking XGB (Top 3)': y_pred_stack_xgb,
    'Bagging CatBoost': y_pred_bagging,
    'Blending (Top 3)': y_pred_blend
}

best_ensemble_pred = best_predictions_map[best_model_name]

# Classification Report
print(f"\nClassification Report - {best_model_name}")
print("-"*80)
print(classification_report(y_test, best_ensemble_pred, 
      target_names=['High (3.5-4.0)', 'Avg (3.0-3.49)', 'Low (<3.0)']))

# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, best_ensemble_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['High (3.5-4.0)', 'Avg (3.0-3.49)', 'Low (<3.0)'],
            yticklabels=['High (3.5-4.0)', 'Avg (3.0-3.49)', 'Low (<3.0)'])
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'../images/best_ensemble_top3_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nConfusion matrix saved to: ../images/best_ensemble_top3_confusion_matrix.png")

print("\n" + "="*80)
print("✅ TOP 3 ENSEMBLE ANALYSIS COMPLETE!")
print("="*80)